In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields



In [ ]:
ss = [
    # {'solution_folder': f"RTS-GMLC_envelope_benchmark_v8.0s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_envelope_benchmark_v9.0s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_envelope_compare_v10.0s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_envelope_compare_v12.0s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_envelope_compare_v9.0su", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_envelope_compare_v11.0su", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_e_reserve_benchmark_v8.0s", 'model_type' : 'e-reserve'},
    {'solution_folder': f"RTS-GMLC_e_reserve_benchmark_v9.0s", 'model_type' : 'e-reserve'},
    {'solution_folder': f"RTS-GMLC_e_reserve_compare_v12.0s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v11.1.0s", 'model_type' : 'envelope'},
    ]

s_uc = []
s_ed = []
gcd_KPI_adequacy = []
gcdi_KPI_adequacy = []
for sol in ss:
    # ρ = sol['ρ']
    s = sol['solution_folder']
    s_uc_name = 's_suc' if sol['model_type'] == 'stochastic' else 's_uc'
    # s_uc_ = load_solutions(s_uc_name, os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_uc.append(s_uc_)
    # s_ed.append(s_ed_)

    gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
    gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], solution_id = s) 

    gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
    gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], solution_id = s)

    gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
    gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

# s_uc = combine_solutions(s_uc)
# s_ed = combine_solutions(s_ed)
gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

if 'µ' in gcdi_KPI_adequacy.columns: 
#         # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
    gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1.0) else x['model_type'], axis=1)
    gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1.0) else x['model_type'], axis=1)




In [ ]:
# values_ = ['reserve_cost_uc','slack_reserve_up_cost_uc','slack_reserve_down_cost_uc','energy_reserve_cost_uc','slack_energy_reserve_up_cost_uc','slack_energy_reserve_down_cost_uc']
# values_ += [
#     'reserve_up_uc_MWh', 'reserve_down_uc_MWh', 'slack_reserve_up_uc_MWh', 'slack_reserve_down_uc_MWh',
#     'required_reserve_up_uc_MWh', 'required_reserve_down_uc_MWh',
#     'energy_reserve_up_uc_MWh', 'energy_reserve_down_uc_MWh', 'slack_energy_reserve_up_uc_MWh', 'slack_energy_reserve_down_uc_MWh',
#     'required_energy_reserve_up_uc_MWh', 'required_energy_reserve_down_uc_MWh'
# ]
# values_ += ['EOV',]
# values_ = [v for v in values_ if v in gcd_KPI_adequacy.columns]
# pivot = pd.pivot_table(
#     gcd_KPI_adequacy,
#     index=['day', 'solution_id'],
#     columns='model_type',
#     values=values_
# )

In [ ]:
from itertools import product

days = [2,3]
scalar = []
scalar_ =pd.DataFrame()
for sol in ss:
    s = sol['solution_folder']
    for day in days:
        try:
            scalar_ = pd.read_parquet(os.path.join("..", "output", s, f'n_{day}','s_ed_scalar.parquet'))
            print(f'(ss, days):{s}, n_{day}')
            scalar_['day'] = day
            scalar_['solution_id'] = sol['solution_folder']
            scalar.append(scalar_)
        except Exception:
            pass
    
    days_str = "-".join(str(d) for d in days)
    try:
        scalar_ = pd.read_parquet(os.path.join("..", "output", s, f'n_{days_str}', 's_ed_scalar.parquet'))
        print(f'(ss, days):{s}, n_{days_str}')
        # scalar_['day'] = 0
        scalar_['solution_id'] = sol['solution_folder']
        scalar.append(scalar_)
    except Exception:
        pass

scalar = pd.concat(scalar)        

In [ ]:
scalar

In [ ]:
gcdi_KPI_adequacy = gcdi_KPI_adequacy.merge(
    scalar[['solution_id', 'configuration', 'day', 'objective_value', 'iteration']],
    on=['solution_id', 'configuration', 'day','iteration'],
    suffixes=('', '_s')
)

In [ ]:
filter_ = (gcdi_KPI_adequacy.iteration == 'demand_1') #&  (gcdi_KPI_adequacy.day == 3)
for k,v in gcdi_KPI_adequacy.loc[filter_,:].groupby([ 'model_type','day', 'µ',]):
    for k2,v2 in v.groupby('model_type'):
        print("")
        print(f"Model Type: {k2}")
        print(v2[['objective_value_uc','objective_value', 'objective_value_s', 'day', 'µ', 'solution_id']])
    

In [ ]:
x = 668553.627217
y = 668256.652730 
print( (x-y)/y)